# HolmHz - Colab Training Template

**He thong Phat hien Anh Tong hop (Synthetic Image Detection)**

Notebook nay dung de train/evaluate model HolmHz tren Google Colab.

## Workflow:
1. Mount Google Drive (luu data + checkpoints)
2. Clone repo & cai dependencies
3. Import `holmhz` package
4. Train / Evaluate / Visualize

## 1. Mount Google Drive

In [ ]:
import os

from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/HolmHz'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive mounted. Working dir: {DRIVE_ROOT}')

## 2. Check GPU

In [ ]:
!nvidia-smi

import torch

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 3. Clone Repo & Install Dependencies

In [ ]:
import os

REPO_DIR = '/content/HolmHz'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/EurusDevSec/HolmHz.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Install holmhz package (editable mode)
!pip install -e . --quiet

import holmhz

print(f'holmhz v{holmhz.__version__} installed')

## 4. Setup W&B

In [ ]:
import wandb

# Option 1: Login bang API key (paste khi duoc hoi)
wandb.login()

# Option 2: Dung Colab secrets (an toan hon)
# from google.colab import userdata
# wandb.login(key=userdata.get('WANDB_API_KEY'))

## 5. Symlink Data tu Google Drive

Thay vi copy data vao Colab, ta tao symlink tu Drive.

**Lan dau**: Upload data len Google Drive theo cau truc:
```
Google Drive/HolmHz/
  data/
    processed/train/
    processed/val/
    manifests/
  outputs/
    checkpoints/
```

In [ ]:
import os

DRIVE_ROOT = '/content/drive/MyDrive/HolmHz'
REPO_DIR = '/content/HolmHz'

links = {
    f'{DRIVE_ROOT}/data': f'{REPO_DIR}/data',
    f'{DRIVE_ROOT}/outputs': f'{REPO_DIR}/outputs',
    f'{DRIVE_ROOT}/weights': f'{REPO_DIR}/weights',
}

for src, dst in links.items():
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst):
        os.remove(dst)
    if not os.path.exists(dst):
        os.symlink(src, dst)
        print(f'{dst} -> {src}')

print('Data linked from Google Drive!')

## 6. Sanity Check

In [ ]:
from pathlib import Path

import timm
import torch

import holmhz

checks = {
    'PyTorch': torch.__version__,
    'CUDA': torch.cuda.is_available(),
    'GPU': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A',
    'timm': timm.__version__,
    'holmhz': holmhz.__version__,
    'data dir': Path('data').exists(),
    'outputs dir': Path('outputs').exists(),
}

for k, v in checks.items():
    print(f'  {k}: {v}')

print('\nReady to train!')

## 7. Training (sau khi co data + model code)

```python
# Option 1: Chay script
!python scripts/train.py --config configs/train.yaml

# Option 2: Import trong notebook
from holmhz.training.trainer import Trainer
from omegaconf import OmegaConf

cfg = OmegaConf.load('configs/train.yaml')
trainer = Trainer(cfg)
trainer.train()
```